# Blender — Predicting Stellar Class

A two-layer submission blender. Metric is **balanced accuracy**; classes are `['GALAXY', 'QSO', 'STAR']`.

**Layer A — probability core (optional).** When out-of-fold and test probabilities of your own
models are available (`oof_*.npy` + `test_*.npy`), a meta stacker (logistic regression) is trained
with an honest OOF balanced-accuracy estimate. This is the only locally-measurable signal.

**Layer B — hard vote over external submissions.** External submissions carry only class labels
(not probabilities). They enter the blend as weighted votes, where each weight equals the
submission's leaderboard score.

**Merge.** On rows where the external submissions agree unanimously (~98.5%), their label is kept.
On the disagreement rows (~1.5%), a weighted vote decides: the probability core (if present) plus
the external votes.

## Inputs and outputs

This notebook is written to run on Kaggle and locally without edits.

- **`stellar_data` dataset** — mirrors `docs/external/`: a `submissions/` folder with one CSV per
  submission (filename = its leaderboard score, e.g. `0.97007.csv`), plus `star_classification.csv`.
  Optionally also `oof_*.npy` / `test_*.npy` for the probability core.
- **Competition dataset** — the folder that contains `train.csv` and `test.csv`. Its `train.csv`
  supplies the ground-truth labels used to validate the probability core.

Paths are auto-detected by scanning `/kaggle/input/*`. When not on Kaggle, the repository layout
under `competitions/predicting-stellar-class/` is used instead.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.metrics import balanced_accuracy_score

CLASSES = ['GALAXY', 'QSO', 'STAR']
C2I = {c: i for i, c in enumerate(CLASSES)}
np.random.seed(42)

KAGGLE_INPUT = Path('/kaggle/input')
ON_KAGGLE = KAGGLE_INPUT.exists()


def find_dir(predicate, roots):
    for root in roots:
        if not root.exists():
            continue
        for p in [root, *sorted(root.rglob('*'))]:
            if p.is_dir() and predicate(p):
                return p
    return None


if ON_KAGGLE:
    search_roots = sorted(KAGGLE_INPUT.iterdir())
    SUBS_DIR = find_dir(lambda p: p.name == 'submissions' and any(p.glob('*.csv')), search_roots)
    assert SUBS_DIR is not None, 'no submissions/ folder found under /kaggle/input'
    STELLAR_DIR = SUBS_DIR.parent
    COMP_DIR = find_dir(lambda p: (p / 'train.csv').exists(), search_roots)
    OUT_DIR = Path('/kaggle/working')
else:
    REPO = Path('competitions/predicting-stellar-class') if Path('competitions').exists() else Path('..')
    STELLAR_DIR = REPO / 'docs' / 'external'
    SUBS_DIR = STELLAR_DIR / 'submissions'
    COMP_DIR = REPO / 'data_processed'
    OUT_DIR = REPO / 'data_processed'

print('on kaggle :', ON_KAGGLE)
print('stellar   :', STELLAR_DIR)
print('subs      :', SUBS_DIR)
print('comp      :', COMP_DIR)
print('out       :', OUT_DIR)

on kaggle : False
stellar   : ../docs/external
subs      : ../docs/external/submissions
comp      : ../data_processed
out       : ../data_processed


## 1. External submissions and their leaderboard-score weights

Every CSV in the `submissions/` folder is one external submission. The filename is parsed as its
leaderboard score and used directly as its voting weight. All submissions are aligned on `id`.

In [2]:
sub_files = sorted(SUBS_DIR.glob('*.csv'))
assert sub_files, f'no submissions found in {SUBS_DIR}'

scores = {f.stem: float(f.stem) for f in sub_files}
subs = {f.stem: pd.read_csv(f).sort_values('id').reset_index(drop=True) for f in sub_files}

ref_id = subs[sub_files[0].stem]['id'].values
for name, df in subs.items():
    assert np.array_equal(df['id'].values, ref_id), f'{name}: id mismatch'

names = list(subs)
print(f'{len(names)} submissions, {len(ref_id)} rows')
for n in names:
    print(f'  {n}: w={scores[n]:.5f}', dict(subs[n]['class'].value_counts()))

L = np.column_stack([subs[n]['class'].map(C2I).values for n in names])
W = np.array([scores[n] for n in names])

5 submissions, 247435 rows
  0.96874: w=0.96874 {'GALAXY': 156899, 'QSO': 51204, 'STAR': 39332}
  0.96973: w=0.96973 {'GALAXY': 157408, 'QSO': 51515, 'STAR': 38512}
  0.96996: w=0.96996 {'GALAXY': 157026, 'QSO': 51391, 'STAR': 39018}
  0.97003: w=0.97003 {'GALAXY': 157186, 'QSO': 51379, 'STAR': 38870}
  0.97007: w=0.97007 {'GALAXY': 157349, 'QSO': 51368, 'STAR': 38718}


## 2. Disagreement region

Rows where every submission already agrees carry no information for blending. The blend can only
ever change a label inside the disagreement region, so it is worth measuring how small that is.

In [3]:
unanimous = np.all(L == L[:, [0]], axis=1)
disagree = ~unanimous
print(f'unanimous rows: {unanimous.sum()} ({unanimous.mean()*100:.2f}%)')
print(f'disagree  rows: {disagree.sum()} ({disagree.mean()*100:.2f}%)')

unanimous rows: 243833 (98.54%)
disagree  rows: 3602 (1.46%)


## 3. Layer A — probability core (only if artifacts are present)

The core looks for `oof_*.npy` (row-aligned to the competition `train.csv`, class order = `CLASSES`)
and matching `test_*.npy`. Each pair is one base model; they are concatenated and fed to a
logistic-regression meta-model. The OOF balanced accuracy is computed honestly via
`cross_val_predict`. Ground-truth labels come from the competition `train.csv`.

If no artifacts are found, the core stays off and the blender runs as a pure weighted hard vote.

In [4]:
npy_roots = [STELLAR_DIR, COMP_DIR, OUT_DIR]
oof_paths, test_paths = [], []
for root in npy_roots:
    if root and root.exists():
        oof_paths += sorted(root.rglob('oof_*.npy'))
        test_paths += sorted(root.rglob('test_*.npy'))

have_probs = bool(oof_paths) and bool(test_paths)
print('probability artifacts:', 'FOUND' if have_probs else 'none -> hard-vote mode')
for p in oof_paths + test_paths:
    print('  ', p)

core_test_proba = None
core_oof_ba = None
meta = meta_oof = y = None

if have_probs:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_predict, StratifiedKFold

    train_csv = COMP_DIR / 'train.csv'
    train_parquet = COMP_DIR / 'train_fe.parquet'
    if train_csv.exists():
        y = pd.read_csv(train_csv, usecols=['class'])['class'].map(C2I).values
    else:
        y = pd.read_parquet(train_parquet)['target'].values

    suffix = lambda p: p.stem.split('_', 1)[1]
    oof_map = {suffix(p): np.load(p) for p in oof_paths}
    test_map = {suffix(p): np.load(p) for p in test_paths}
    keys = [k for k in oof_map if k in test_map]
    assert keys, 'no oof_*/test_* pairs with a shared suffix'
    print('models in stack:', keys)

    for k in keys:
        assert oof_map[k].shape[0] == len(y), f'{k}: oof rows {oof_map[k].shape} != {len(y)}'
        assert test_map[k].shape[0] == len(ref_id), f'{k}: test rows != {len(ref_id)}'

    Xoof = np.hstack([oof_map[k] for k in keys])
    Xtest = np.hstack([test_map[k] for k in keys])

    meta = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial')
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    meta_oof = cross_val_predict(meta, Xoof, y, cv=skf, method='predict_proba')
    core_oof_ba = balanced_accuracy_score(y, meta_oof.argmax(1))
    print(f'\nprobability core OOF balanced accuracy = {core_oof_ba:.5f}')

    meta.fit(Xoof, y)
    core_test_proba = meta.predict_proba(Xtest)
    assert list(meta.classes_) == list(range(3)), f'unexpected class order: {meta.classes_}'

probability artifacts: none -> hard-vote mode


## 4. Merge the layers into the final blend

For each row a weighted vote is accumulated across the three classes:

- every external submission adds its weight `W[i]` to the class it predicted;
- the probability core (if present) adds `CORE_WEIGHT * proba[class]` as a soft vote.

`CORE_WEIGHT` defaults to the sum of external weights, giving the locally-validated core a say
comparable to the whole external panel. Setting it to `0` recovers a pure external hard vote.
The argmax over the accumulated votes is the final label.

In [5]:
CORE_WEIGHT = float(W.sum()) if core_test_proba is not None else 0.0
print(f'CORE_WEIGHT = {CORE_WEIGHT:.4f} | sum of external weights = {W.sum():.4f}')

n = len(ref_id)
votes = np.zeros((n, 3), dtype=np.float64)
for j in range(len(names)):
    np.add.at(votes, (np.arange(n), L[:, j]), W[j])
if core_test_proba is not None:
    votes += CORE_WEIGHT * core_test_proba

blend_idx = votes.argmax(1)
blend_lbl = np.array(CLASSES)[blend_idx]

best_name = max(names, key=lambda x: scores[x])
best_idx = L[:, names.index(best_name)]
print(f'\nbest external submission: {best_name}')
print(f'blend differs from it on {(blend_idx != best_idx).sum()} rows')
print('all changes inside disagreement region:', bool(np.all(disagree[blend_idx != best_idx])))
print('blend class distribution:', dict(zip(*np.unique(blend_lbl, return_counts=True))))

CORE_WEIGHT = 0.0000 | sum of external weights = 4.8485

best external submission: 0.97007
blend differs from it on 249 rows
all changes inside disagreement region: True
blend class distribution: {'GALAXY': 157202, 'QSO': 51376, 'STAR': 38857}


## 5. Optional per-class thresholds over the probability core

The project's weak spot is STAR recall. With a probability core available, per-class multipliers
can shift the decision boundary toward STAR. The multipliers are tuned on OOF (honestly) and then
applied to the test votes. Disabled by default — enable deliberately and validate by submitting.

In [6]:
APPLY_THRESHOLDS = False

if APPLY_THRESHOLDS and core_test_proba is not None:
    from itertools import product
    grid = np.linspace(0.8, 1.4, 13)
    best_ba, best_m = core_oof_ba, (1.0, 1.0, 1.0)
    for mg, mq, ms in product(grid, grid, grid):
        m = np.array([mg, mq, ms])
        ba = balanced_accuracy_score(y, (meta_oof * m).argmax(1))
        if ba > best_ba:
            best_ba, best_m = ba, (mg, mq, ms)
    print(f'multipliers {best_m} -> OOF BA {best_ba:.5f} (was {core_oof_ba:.5f})')
    m = np.array(best_m)
    blend_idx = (votes * m).argmax(1)
    blend_lbl = np.array(CLASSES)[blend_idx]
    print('new class distribution:', dict(zip(*np.unique(blend_lbl, return_counts=True))))
else:
    print('thresholds not applied')

thresholds not applied


## 6. Write the submission

The result is written to the working directory as `submission.csv` (`id,class`), ready to submit.

In [7]:
sub = pd.DataFrame({'id': ref_id, 'class': blend_lbl})
assert sub['class'].isin(CLASSES).all()
out_path = OUT_DIR / 'submission.csv'
sub.to_csv(out_path, index=False)
print('written:', out_path, sub.shape)
sub.head()

written: ../data_processed/submission.csv (247435, 2)


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
